## Overview of Assignment 4

This assignment focuses on exploring and implementing advanced concepts and techniques in information retrieval. The primary objectives are to build Retrieval Augumentation Generation, and learn about Language Models

## Enter your details below

## Name

Akash Maity

## Banner ID

B00921683

## GitHub Link of your Assingment 4

https://github.com/Akash324-dotcom/RL-Paper-RAG

## Q1 : Setting up the libraries and the environment

In [15]:
!pip install -q arxiv pypdf langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu transformers torch huggingface_hub accelerate

In [16]:
import arxiv
import pypdf
import langchain
import sentence_transformers
import faiss
import transformers
import torch

print("arxiv:", arxiv.__version__)
print("langchain:", langchain.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

arxiv: 4.0.1
langchain: 1.3.13
sentence_transformers: 5.6.0
transformers: 5.13.1
torch: 2.11.0+cpu
CUDA available: False


## Q2:  Data Preprocessing and Model Selection

In [17]:
import arxiv
import os
import time
from urllib.request import urlretrieve

PDF_DIR = "arxiv_pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# I am using only 50 pdfs to save space and fast processing.
NUM_PAPERS = 50

client = arxiv.Client()
search = arxiv.Search(
    query='cat:cs.LG AND ("reinforcement learning")',
    max_results=NUM_PAPERS,
    sort_by=arxiv.SortCriterion.SubmittedDate,
)

papers_metadata = []

for i, result in enumerate(client.results(search)):
    safe_id = result.get_short_id().replace("/", "_")
    pdf_path = os.path.join(PDF_DIR, f"{safe_id}.pdf")

    if not os.path.exists(pdf_path):
        urlretrieve(result.pdf_url, pdf_path)
        time.sleep(1)

    papers_metadata.append({
        "id": safe_id,
        "title": result.title,
        "authors": [a.name for a in result.authors],
        "published": str(result.published),
        "pdf_path": pdf_path,
        "abstract": result.summary,
    })

    print(f"[{i+1}/{NUM_PAPERS}] Downloaded: {result.title[:70]}...")

print(f"\nDone. Downloaded {len(papers_metadata)} papers to '{PDF_DIR}/'")

[1/50] Downloaded: RRC: Unlocking Generative Reward Models in LLM Reinforcement Learning ...
[2/50] Downloaded: Stochastic Dynamics on Persistence Diagram Space via Reinforcement Lea...
[3/50] Downloaded: Does Latent Context Help? A Controlled Evaluation of Inverse Reinforce...
[4/50] Downloaded: Hybrid-Adaptive Thread Tuning to Mitigate Simulation Execution Bottlen...
[5/50] Downloaded: ProDVI: Programmatic Dynamics Priors for Value Network Initialization...
[6/50] Downloaded: Observation-Grounded Self-Predictive Reinforcement Learning for Visual...
[7/50] Downloaded: AgentOPSD: Recursive Self-Distillation for Agentic Reinforcement Learn...
[8/50] Downloaded: Training a Conditioned Video Game Agent on a VLM Annotated Dataset...
[9/50] Downloaded: VLMs for Videogame Data Annotation...
[10/50] Downloaded: On-Policy Delta Distillation for Multilingual Math Reasoning...
[11/50] Downloaded: LC-GRPO: Bridging Train-Inference Gap for Flow-Based GRPO with Langevi...
[12/50] Downloaded: IFlowN

In [18]:
# sample paper, first one
paper = papers_metadata[0]
print(f"Title: {paper['title']}")
print(f"Authors: {', '.join(paper['authors'])}")
print(f"Published: {paper['published']}")
print(f"Abstract: {paper['abstract'][:200]}...")

Title: RRC: Unlocking Generative Reward Models in LLM Reinforcement Learning via Ranking-Based Reward Construction
Authors: Chenglong Wang, Ziming Zhu, Yifu Huo, Bei Li, Qiaozhi He, Yan Ding, Xiaoyang Hao, Yuxin Gao, Tianhua Zhou, Xiaojia Chang, Tongran Liu, Jingbo Zhu
Published: 2026-08-06 17:24:36+00:00
Abstract: Recent advances in reward modeling show a paradigm shift from discriminative reward models to generative reward models. However, despite their strong capabilities in response ranking, generative rewar...


But when some query i will run, running it over all pdfs one-by-one would be really stretchy. So i am gonna break down after every 500 characters.

https://pypdf.readthedocs.io/en/latest/modules/PageObject.html - this is the file I used for PdfReader

In [19]:
from pypdf import PdfReader
import re

def extract_text_from_pdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    except Exception as e:
        print(f"Failed to read {pdf_path}: {e}")
        return ""

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for paper in papers_metadata:
    paper["full_text"] = clean_text(extract_text_from_pdf(paper["pdf_path"]))

non_empty = [p for p in papers_metadata if len(p["full_text"]) > 500]
print(f"Papers with usable extracted text: {len(non_empty)} / {len(papers_metadata)}")
print("\nSample (first 500 chars of paper 0):\n")
print(papers_metadata[0]["full_text"][:500])

Papers with usable extracted text: 50 / 50

Sample (first 500 chars of paper 0):

arXiv preprint RRC: UNLOCKINGGENERATIVEREWARDMODELS INLLM REINFORCEMENTLEARNING VIARANKING- BASEDREWARDCONSTRUCTION Chenglong Wang1,2, Ziming Zhu1, Yifu Huo1, Bei Li1, Qiaozhi He1, Yan Ding1, Xiaoyang Hao1, Yuxin Gao1, Tianhua Zhou3, Xiaojia Chang3, Tongran Liu4, Jingbo Zhu1,2, Zhengtao Yu5, Tong Xiao1,2∗ 1School of Computer Science and Engineering, Northeastern University, Shenyang, China 2NiuTrans Research, Shenyang, China 3Independent Researcher, Beijing, China 4CAS Key Laboratory of Behavior


In [20]:
print(papers_metadata[0]["full_text"][500:1200])


al Science, Institute of Psychology, CAS, Beijing, China 5Kunming University of Science and Technology {wangchenglong, xiaotong}@mail.neu.edu.cn ABSTRACT Recent advances in reward modeling show a paradigm shift from discriminative reward models to generative reward models. However, despite their strong ca- pabilities in response ranking, generative reward models have not realized their potential in reinforcement learning (RL). Our analysis reveals that this limitation arises from amismatchbetween the comparative nature of generative reward mod- eling and the scalar scoring paradigm adopted by existing RL algorithms. To bridge this gap, we propose aR anking-basedR ewardC onstruction (RRC) ap-


In [21]:
from transformers import AutoTokenizer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

sample_text = papers_metadata[0]["full_text"][:300]
tokens = tokenizer.tokenize(sample_text)

print("Sample text:\n", sample_text)
print(f"\nNumber of tokens: {len(tokens)}")
print("First 20 tokens:", tokens[:20])

Sample text:
 arXiv preprint RRC: UNLOCKINGGENERATIVEREWARDMODELS INLLM REINFORCEMENTLEARNING VIARANKING- BASEDREWARDCONSTRUCTION Chenglong Wang1,2, Ziming Zhu1, Yifu Huo1, Bei Li1, Qiaozhi He1, Yan Ding1, Xiaoyang Hao1, Yuxin Gao1, Tianhua Zhou3, Xiaojia Chang3, Tongran Liu4, Jingbo Zhu1,2, Zhengtao Yu5, Tong Xi

Number of tokens: 109
First 20 tokens: ['ar', '##xi', '##v', 'prep', '##rin', '##t', 'rr', '##c', ':', 'unlock', '##ing', '##gen', '##erative', '##rew', '##ard', '##mo', '##del', '##s', 'in', '##ll']


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

all_chunks = []
for paper in papers_metadata:
    if len(paper["full_text"]) < 200:
        continue
    for chunk in text_splitter.split_text(paper["full_text"]):
        all_chunks.append({
            "text": chunk,
            "paper_id": paper["id"],
            "paper_title": paper["title"],
        })

print(f"Total chunks created from {len(papers_metadata)} papers: {len(all_chunks)}")
print(f"\nExample chunk:\n{all_chunks[0]['text'][:300]}")
print(f"\n(from paper: {all_chunks[0]['paper_title']})")

Total chunks created from 50 papers: 7780

Example chunk:
arXiv preprint RRC: UNLOCKINGGENERATIVEREWARDMODELS INLLM REINFORCEMENTLEARNING VIARANKING- BASEDREWARDCONSTRUCTION Chenglong Wang1,2, Ziming Zhu1, Yifu Huo1, Bei Li1, Qiaozhi He1, Yan Ding1, Xiaoyang Hao1, Yuxin Gao1, Tianhua Zhou3, Xiaojia Chang3, Tongran Liu4, Jingbo Zhu1,2, Zhengtao Yu5, Tong Xi

(from paper: RRC: Unlocking Generative Reward Models in LLM Reinforcement Learning via Ranking-Based Reward Construction)


In [23]:
from collections import Counter

chunk_counts = Counter(c["paper_id"] for c in all_chunks)
counts_list = list(chunk_counts.values())

print(f"Papers represented: {len(chunk_counts)}")
print(f"Min chunks from a single paper: {min(counts_list)}")
print(f"Max chunks from a single paper: {max(counts_list)}")
print(f"Average chunks per paper: {sum(counts_list)/len(counts_list):.1f}")

biggest_id = max(chunk_counts, key=chunk_counts.get)
biggest_title = next(p["title"] for p in papers_metadata if p["id"] == biggest_id)
print(f"\nLongest paper by chunk count: '{biggest_title}' with id: {biggest_id}")

Papers represented: 50
Min chunks from a single paper: 49
Max chunks from a single paper: 365
Average chunks per paper: 155.6

Longest paper by chunk count: 'LLM Serving in the Wild: An Empirical Study of Frameworks, Methods, and System Designs' with id: 2608.03036v1


In [24]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embedding_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

documents = [
    Document(page_content=chunk["text"], metadata={
        "paper_id": chunk["paper_id"],
        "paper_title": chunk["paper_title"],
    })
    for chunk in all_chunks
]

vector_store = FAISS.from_documents(documents, embedding_model)
vector_store.save_local("faiss_rl_papers_index")

print(f"Vector store built with {len(documents)} chunks.")
print("Saved to 'faiss_rl_papers_index/'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built with 7780 chunks.
Saved to 'faiss_rl_papers_index/'


In [25]:
from collections import Counter
chunk_counts = Counter(c["paper_id"] for c in all_chunks)

def search_with_attribution(query, k=5):
    results = vector_store.similarity_search_with_score(query, k=k)
    print(f"Query: '{query}'\n")
    for i, (doc, score) in enumerate(results):
        title = doc.metadata["paper_title"]
        pid = doc.metadata["paper_id"]
        total_chunks_from_this_paper = chunk_counts[pid]
        print(f"Result {i+1} | similarity score: {score:.4f}")
        print(f"From paper: {title[:70]}")
        print(f"(This paper contributed {total_chunks_from_this_paper} chunks total to the index)")
        print(f"Snippet: {doc.page_content[:200]}...")
        print()

search_with_attribution("What are the main challenges in reward shaping for reinforcement learning?")


Query: 'What are the main challenges in reward shaping for reinforcement learning?'

Result 1 | similarity score: 0.7719
From paper: Reward Structure Shapes the Interaction Between Episodic Exploration a
(This paper contributed 282 chunks total to the index)
Snippet: 2022. Reward Machines: Exploiting Reward Function Structure in Reinforcement Learning.Journal of Artificial Intelligence Research (JAIR), 73: 173–208. Toro Icarte, R.; Waldie, E.; Klassen, T. Q.; Vale...

Result 2 | similarity score: 0.8578
From paper: Training a Conditioned Video Game Agent on a VLM Annotated Dataset
(This paper contributed 49 chunks total to the index)
Snippet: to the desired returns and we discuss the difficulties and limitations that emerged in our early experiments. Index Terms—Vision Language Models, Reinforcement Learn- ing, video games, Agents, Conditi...

Result 3 | similarity score: 0.8666
From paper: Reward Structure Shapes the Interaction Between Episodic Exploration a
(This paper contributed 2

In [26]:
search_with_attribution("How does reinforcement learning improve performance?")

Query: 'How does reinforcement learning improve performance?'

Result 1 | similarity score: 0.8274
From paper: CVPO: Enhancing LLM Reinforcement Learning Reasoning via Value-Varianc
(This paper contributed 86 chunks total to the index)
Snippet: the upper bound of model performance. Empirical results further confirm that this method significantly enhances both the accuracy and robustness of the model. • The evaluation results on multiple math...

Result 2 | similarity score: 0.8479
From paper: Foundations of Reinforcement Learning and Control:Connections and New 
(This paper contributed 206 chunks total to the index)
Snippet: Prentice Hall, 1989. [69] I. Osband and B. Van Roy. Why is posterior sampling better than optimism for reinforcement learning? In International Conference on Machine Learning, pages 2701–2710, 2017. [...

Result 3 | similarity score: 0.8637
From paper: Reward Structure Shapes the Interaction Between Episodic Exploration a
(This paper contributed 282 chunks total to

## Q3: Implementing RAG using LangChain for different queries

A RAG (Retrieval-Augmented Generation) pipeline has three main components:
-  a retriever, which converts the user's question into a vector and searches a vector store (FAISS, in our case) for the most similar text chunks.

-  a prompt, which combines the retrieved chunks with the original question into a structured instruction for the language mode to undertand ckearly.

-  a generator (the LLM: TinyLlama-1.1B-Chat in our case), which reads that combined prompt and produces a natural-language answer grounded in the retrieved context rather than purely from its own training data.

This design reduces hallucination, since the model is explicitly instructed to answer using the provided evidence.

In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    dtype=torch.float32,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print(f"Loaded {LLM_MODEL_ID} on {device}")
print(f"Model has {sum(p.numel() for p in llm_model.parameters())/1e9:.2f}B parameters")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cpu
Model has 1.10B parameters


### Q3.2 Choosing a pretrained language model

**Model chosen: TinyLlama/TinyLlama-1.1B-Chat-v1.0**

**Justification:** TinyLlama is a 1.1-billion-parameter causal language model, fine-tuned
specifically for chat-style instruction following. It was selected due to compute constrains since I am not using GPU and this makes running only 1.1 B parfameters easy.
The tuning of this model is
   fine-tuned to follow structured chat prompts (using `<|system|>`, `<|user|>`,
   `<|assistant|>` tags)

In [28]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""<|system|>
You are a helpful research assistant. Answer the question using ONLY the context provided below. If the answer is not contained in the context, say you don't know rather than guessing.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
""")

print(prompt_template.template)

<|system|>
You are a helpful research assistant. Answer the question using ONLY the context provided below. If the answer is not contained in the context, say you don't know rather than guessing.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>



In [29]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

generation_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [30]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # retrieve top 3 chunks

def format_docs(docs):
    return "\n\n".join(
        f"[From: {doc.metadata['paper_title']}]\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# This would make the RAG chain assembled

In [31]:
question = "What are the main challenges in reward shaping for reinforcement learning?"

answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:", answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: What are the main challenges in reward shaping for reinforcement learning?

Answer: The main challenges in reward shaping for reinforcement learning are:

1. Limited rewards: Reinforcement learning algorithms typically have limited rewards, which can make it difficult to learn policies that maximize rewards.

2. Noisy rewards: Rewards can be noisy, which can make it difficult to learn policies that are consistent with the true reward function.

3. Reward shaping: Reward shaping involves modifying the rewards to make them more consistent with the desired reward function. This can be challenging because it requires modifying the reward function itself.

4. Limited data: Reward shaping requires a large amount of data to train the reward function, which can be challenging to obtain given the limited availability of data in many real-world applications.

5. Non-stationary environments: Reinforcement learning algorithms are typically designed for stationary environments, but some e

In [32]:
questions = [
    "What is Proximal Policy Optimization and how does it work?",
    "How do actor-critic methods reduce variance in policy gradient estimation?",
]

for q in questions:
    print("Question:", q)
    answer = rag_chain.invoke(q)
    print("\nAnswer:", answer)
    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is Proximal Policy Optimization and how does it work?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Proximal Policy Optimization (PPO) is a popular reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability. It works by limiting excessive policy changes during updates using a clipped objective function, preventing significant deviations from the previous policy and avoiding the loss of training progress. PPO is a proximal policy optimization algorithm that uses a proximal operator to approximate the gradient of the objective function. The proximal operator is a function that maps a function f(θ) to a function f(θ) + λθ, where λ is a scalar parameter. The proximal operator is used to approximate the gradient of the objective function with a lower-order function that is less sensitive to small changes in the policy. This approach helps to prevent the loss of training progress and maintains stability during updates.

Question: How do actor-critic methods reduce variance in policy gradient estimation?

Answer: Th

### Q3.5  

- **Reward shaping query:** The model produced a well-structured list of 4 challenges
  (limited reward function data, non-stationarity, limited capacity, data scarcity), showing it
  can form the retrieved chunks into an organized answer.

- **PPO query:** The answer correctly described PPO's core mechanism (clipped objective
  function, policy stability), but also cited niche applications ("tax-aware RL," "job shop
  scheduling") that appear to be actual paper titles from the retrieved corpus - overgeneralizing.

- **Actor-critic query:** The model explicitly stated the retrieved context did not contain
  information to answer the question, rather than guessing - direct evidence that the "say you
  don't know" instruction in our prompt template works as intended.

- **Overall relevance:** All three answers used terminology and concepts that go back to the
  actual chunks retrieved one (not generic textbook knowledge pulled from the model's own training),
  confirming the retrieval step is meaningfully influencing generation.


## Q4 : Modify and evaluate the different components of RAG

### Q4.1

I am choosing **Query Expansion**: having the LLM generate alternative phrasings of a query before
retrieval, so we can search FAISS with multiple angles of the same question instead of just
one single phrasing.

We first try a simple instruction, directly asking TinyLlama to generate two alternative phrasings of the query. This tests whether the model can follow a basic rewriting instruction without any additional guidance or examples or no.

In [33]:
def expand_query(original_query, model, tokenizer, device):

    prompt = f"""<|system|>
        You are a helpful assistant. Generate two alternative versions of the given search query.
        The goal is to create variations that might help retrieve relevant information from research papers.
        Only list the alternative queries, one per line. Do not include any kind of explanations, numbering, or extra text.
      </s>
      <|user|>
        Original query: {original_query}
      </s>
      <|assistant|>
      """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Split into lines, clean up, drop empty ones
    variations = [line.strip("-• \n") for line in generated_text.split("\n") if line.strip()]
    variations = [v for v in variations if len(v) > 5][:2]  # keep at most 2 usable variations

    return [original_query] + variations


# Test this
test_query = "What are the main challenges in reward shaping for reinforcement learning?"
expanded = expand_query(test_query, llm_model, llm_tokenizer, device)

print("Original query:", test_query)
print("\nExpanded queries:")
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original query: What are the main challenges in reward shaping for reinforcement learning?

Expanded queries:
  1. What are the main challenges in reward shaping for reinforcement learning?
  2. Correct: What are the main challenges in reward shaping for reinforcement learning?
  3. Alternative query 1: How can we identify and address challenges in reinforcement learning that hinder the development of effective reward shaping algorithms?


This needs filtering the junk lines like "Original query" and "Alternative query 1". To fix this, we add an example, showing the model one worked example of a query being rephrased.

In [34]:
def expand_query(original_query, model, tokenizer, device):
    prompt = f"""<|system|>
You rewrite search queries. Given a query, output exactly two rephrased versions, one per line, with no extra text, no labels, no numbering.
</s>
<|user|>
Query: What causes model overfitting?
</s>
<|assistant|>
Why do machine learning models overfit?
What factors lead to overfitting in models?
</s>
<|user|>
Query: {original_query}
</s>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Filtering out the junk lines
    bad_starts = ("original", "query:", "two alternative", "alternative", "-", "•")
    variations = []
    for line in generated_text.split("\n"):
        line = line.strip("-• \n")
        if len(line) > 10 and not line.lower().startswith(bad_starts):
            variations.append(line)

    return [original_query] + variations[:2]


test_query = "What are the main challenges in reward shaping for reinforcement learning?"
expanded = expand_query(test_query, llm_model, llm_tokenizer, device)

print("Original query:", test_query)
print("\nExpanded queries:")
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original query: What are the main challenges in reward shaping for reinforcement learning?

Expanded queries:
  1. What are the main challenges in reward shaping for reinforcement learning?
  2. The main challenges in reward shaping for reinforcement learning are:
  3. 1. Scalability: As the number of states and actions increases, the number of parameters in the model grows exponentially, making it difficult to perform the inverse reinforcement learning (IRL) task accur


Thsi caused the version to drift.. so i will make it more disciplined by making it short.

In [35]:
def expand_query(original_query, model, tokenizer, device):
    prompt = f"""<|system|>
You rewrite search queries as questions. Given a query, output exactly two rephrased QUESTIONS, one per line. Each line MUST end with a question mark. Do not explain anything. Do not answer the question.
</s>
<|user|>
Query: What causes model overfitting?
</s>
<|assistant|>
Why do machine learning models overfit?
What factors lead to overfitting in models?
</s>
<|user|>
Query: {original_query}
</s>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,   # tight to force short output, less room to drift into explanation
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    bad_starts = ("original", "query:", "two alternative", "alternative", "-", "•")
    variations = []
    for line in generated_text.split("\n"):
        line = line.strip("-• \n")
        # Require it to actually look like a question now
        if len(line) > 10 and line.endswith("?") and not line.lower().startswith(bad_starts):
            variations.append(line)

    return [original_query] + variations[:2]


test_query = "What are the main challenges in reward shaping for reinforcement learning?"
expanded = expand_query(test_query, llm_model, llm_tokenizer, device)

print("Original query:", test_query)
print("\nExpanded queries:")
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")


[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original query: What are the main challenges in reward shaping for reinforcement learning?

Expanded queries:
  1. What are the main challenges in reward shaping for reinforcement learning?


Having documented query expansion's limitations, modifying the prompt template itself is important so I define two versions, our original Q3 prompt, and a stricter version that explicitly does not allow guessing unstated facts, to directly test whether prompt-level instructions reduce hallucination  or not.

In [36]:
# Original prompt (baseline, from Q3)
original_prompt = PromptTemplate.from_template("""<|system|>
You are a helpful research assistant. Answer the question using ONLY the context provided below. If the answer is not contained in the context, say you don't know rather than guessing.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
""")

# Improved prompt: more explicit grounding + anti-hallucination instruction
improved_prompt = PromptTemplate.from_template("""<|system|>
You are a careful research assistant. Answer the question using ONLY facts explicitly stated in the context below.
Do NOT guess names, dates, or attributions that are not directly stated in the context.
If a detail (such as who created something) is not explicitly stated in the context, say "the context does not specify this" instead of guessing.
Keep your answer concise and avoid repeating the same point multiple times.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
""")

In [37]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Toa ddress the looping issue i am adding the repetation penalty
improved_generation_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.3,   # penalizes the model for repeating itself
    return_full_text=False,
)

improved_llm = HuggingFacePipeline(pipeline=improved_generation_pipeline)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(
        f"[From: {doc.metadata['paper_title']}]\n{doc.page_content}"
        for doc in docs
    )

# Improved chain: new prompt + repetition penalty
improved_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | improved_prompt
    | improved_llm
    | StrOutputParser()
)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [38]:
ppo_question = "What is Proximal Policy Optimization and how does it work?"

print("BASELINE (original prompt, no repetition penalty)")
baseline_answer = rag_chain.invoke(ppo_question)
print(baseline_answer)

print("\n\nIMPROVED (stricter prompt + repetition penalty)")
improved_answer = improved_rag_chain.invoke(ppo_question)
print(improved_answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASELINE (original prompt, no repetition penalty)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Proximal Policy Optimization (PPO) is a reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability. Its core mechanism is a clipped objective function that limits excessive policy changes during updates. PPO works by updating the actor networkθ using a proximal policy gradient (PPG) algorithm, which is a variant of the policy gradient algorithm that uses a proximal operator to approximate the gradient. The proximal operator is a function that maps a function f(θ) to a function f(θ) + λ(θ)∇f(θ), where λ is a scalar parameter and f(θ) is the function to be optimized. The proximal operator is used to approximate the gradient of the objective function, which is the reward function R(θ) in the PPO algorithm. The PPG algorithm updatesθ by minimizing the difference between the proximal PPG(θ) and the original PPG(


IMPROVED (stricter prompt + repetition penalty)
Proximal Policy Optimization (PPO) is a reinforcement learning a

In [39]:
for trial in range(2):
    print(f"TRIAL {trial + 2}")  # trial 1 was our first run above

    print("\nBASELINE")
    print(rag_chain.invoke(ppo_question))

    print("\nIMPROVED")
    print(improved_rag_chain.invoke(ppo_question))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TRIAL 2

BASELINE


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Proximal Policy Optimization (PPO) is a popular reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability. The core mechanism of PPO is to limit excessive policy changes during updates using a clipped objective function. This function is designed to prevent significant deviations from the previous policy and avoid the loss of training stability.

PPO works by iteratively updating the policy parameter θ using a gradient descent algorithm. The gradient descent step involves computing the gradient of the objective function with respect to θ, which is then used to update the policy parameter. The objective function is a function that measures the performance of the policy in terms of the expected reward or loss.

The clipped objective function used in PPO is a function that clips the gradient of the objective function at a certain threshold. This threshold is determined based on the target value of the objective function, 

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Proximal Policy Optimization (PPO), also known as value-variance adaptation (VVA), is an updated version of traditional reinforcement learning methods such as deep deterministic policies gradient (DDPG) and stochastic temporal difference (STD) which improves policy performance without requiring additional computationally expensive steps like target networks or experience replay buffer. It works by updating the critic's estimate of the action values based on current observations and past experiences rather than just using them alone. This approach helps reduce exploratory behavior and improve generalization capabilities when dealing with complex environments. In PPO, the critic uses a weighted combination of the original value estimates and the estimated variance of those values to update its parameters. By doing so, the system learns to adapt more quickly to new situations and make better decisions overall. Overall, PPO reduces the need for manual intervention by allowing agents to lea

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Proximal Policy Optimization (PPO) is a popular reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability. It works by limiting excessive policy changes during updates using a clipped objective function. The algorithm's core mechanism is a proximal policy optimization (PPO) algorithm, which is a variant of the policy optimization algorithm that uses a proximal operator to approximate the gradient of the objective function.

PPO works by iteratively updating the policy parameters θ using a gradient descent step with a proximal operator Φ, which is a proximal operator that approximates the gradient of the objective function. The proximal operator Φ is applied to the gradient of the objective function, which is then used to update the policy parameters θ. This process is repeated until the policy parameters converge to a stable state.

The proximal operator Φ is chosen based on the objective function and the learning rate

Trial 2 has a hallucination - "Google Brain Researchers Radford et al", this is not true as Alec Radford is known for GPT and CLIP work in OpenAI not Google Brain.

**Q4.3**

In [40]:
reward_question = "What are the main challenges in reward shaping for reinforcement learning?"

for k in [1, 3, 6]:
    print(f"k = {k}")

    test_retriever = vector_store.as_retriever(search_kwargs={"k": k})
    test_chain = (
        {"context": test_retriever | format_docs, "question": RunnablePassthrough()}
        | improved_prompt
        | improved_llm
        | StrOutputParser()
    )

    answer = test_chain.invoke(reward_question)
    print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


k = 1


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The main challenges in reward shaping for reinforcement learning include:

1. Limited observation space - The number of possible actions an agent can take depends on its environment, which limits how much information it has to learn about rewards from experience. This makes it difficult to accurately estimate future rewards based solely on past observations.

2. Nonlinearity - Many environments have nonlinear dynamics, making it hard to predict exactly what will happen next when taking action. For example, if an agent's goal is to reach a specific location within a maze, there may be many different paths through the maze with varying degrees of difficulty. Estimating these complex trajectories requires more sophisticated algorithms than simple linear models.

3. Uncertainty - Environmental uncertainties such as noise, randomness, and sensor errors can affect the accuracy of estimated rewards. These factors make it harder to precisely model the relationship between observed behavior
k =

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The main challenges in reward shaping for reinforcement learning include:

1. Limited observation space: The number of possible actions available to the agent can be limited by the size of the state space. This makes it difficult to create complex behaviors without resorting to cheating techniques such as memorizing past states.

2. Uncertainty about future rewards: Even if there is enough information about the current state and action taken, uncertainty remains regarding whether the next step will result in a positive or negative outcome. This adds another layer of complexity to the problem.

3. Nonlinearity in reward function: Some environments may have nonlinearities in their reward functions which make them more difficult to predict accurately. For example, some puzzle-solving tasks require precise timing between inputs and outputs, while others involve manipulating objects over time.

4. Model complexity: As the number of parameters required to represent the entire policy network 

We tested k=1, 3, and 6 (how many chunks the model sees) on the same question. All three
answers looked similar in quality, none of them used the specific paper wording we saw in
earlier tests. This suggests that changing k alone didn't clearly make answers better or worse
here.

Might be because the model uses randomness when generating text (temperature=0.7), so the
same settings can give different results each time it runs.

So running each k multiple times would give a more
reliable answer.

### Q4.4

We changed two things: added a repetition penalty, and used a stricter "don't guess" prompt.

- **Repetition penalty worked well.** In Q3, the model sometimes got stuck repeating the same
  phrase over and over. After adding """repetition_penalty=1.3""", this didn't happen again in any
  of our tests.

- **The stricter prompt did NOT stop hallucination.** Across multiple runs, the "improved"
  version still made up fake facts about who created PPO, claiming it was invented by
  "OpenAI's Sam Altman," by "Google Brain researcher Radford," and calling it "proximal policy
  iteration" (not a real name). The baseline version, without the strict prompt, avoided these
  mistakes more often.

- **Changing k (number of retrieved chunks) didn't show a clear pattern.** Answers looked
  similar whether we used k=1, 3, or 6.

Therfore, small fixes like repetition penalty work reliably, because they directly control
how the model generates text. But telling the model "don't guess" in the prompt isn't enough to
stop it from making up false facts, the prompt just is nit enough to fix that.

## Q5: Selecting and implementing a pretrained model for a new task

Q5.1

I seelcted **hallucination/faithfulness detection** which is used to check whether the retrieved text is actually supported or not or is it just fabricated.

The hallucinations we found earlier, it can be used to check and closing the loop by making an automated detector for it.

NLI (Natural Language Inference) = the model checks if one sentence (a claim) is actually supported by another sentence (some evidence).

Q5.2

I chose "facebook/bart-large-mnli" a BART model fine-tuned via Supervised
Fine-Tuning (SFT) on the MultiNLI dataset for Natural Language Inference (entailment /
contradiction / neutral classification).

I was inspired by the research on NLI style entailment used by Google TRUE benchmark so tried this.

This model is good as an encoder-decoder model used for
*classification* via NLI, rather than free-form text generation.

In [41]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL_ID = "facebook/bart-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID)

print("Label mapping:", nli_model.config.id2label)
print("Model loaded successfully.")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Label mapping: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
Model loaded successfully.


Label mapping confirmed: 0=contradiction, 1=neutral, 2=entailment

In [46]:
import re

def split_into_sentences(text):
    """Simple sentence splitter."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if len(s) > 15]  # skip tiny fragments

def check_faithfulness(context, answer):
    """
    For each sentence in the answer, check if it's entailed by the context.
    """
    sentences = split_into_sentences(answer)
    results = []

    for sentence in sentences:
        inputs = nli_tokenizer(context, sentence, return_tensors="pt", truncation=True, max_length=1024)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]

        label_id = torch.argmax(probs).item()
        label = nli_model.config.id2label[label_id]
        confidence = probs[label_id].item()

        results.append((sentence, label, confidence))

    return results

In [44]:
ppo_question = "What is Proximal Policy Optimization and how does it work?"

# Get the actual retrieved context (same as what the RAG chain used above)
retrieved_docs = retriever.invoke(ppo_question)
context_text = format_docs(retrieved_docs)

# The hallucinated answer that we earlier got
hallucinated_answer = """Proximal Policy Optimization (PPO) is an adaptive decentralized policy gradient method developed by OpenAI's Sam Altman and others. It works by limiting the rate at which policies change during each update step based on past experience, allowing for more stable convergence towards optimal solutions."""

results = check_faithfulness(context_text, hallucinated_answer)

print("Question:", ppo_question)
for sentence, label, confidence in results:
    flag = "POSSIBLE HALLUCINATION" if label != "entailment" else "supported"
    print(f"[{label.upper()} ({confidence:.2f})] {flag}")
    print(f"  \"{sentence}\"\n")

Question: What is Proximal Policy Optimization and how does it work?
[NEUTRAL (0.97)] POSSIBLE HALLUCINATION
  "Proximal Policy Optimization (PPO) is an adaptive decentralized policy gradient method developed by OpenAI's Sam Altman and others."

[ENTAILMENT (0.91)] supported
  "It works by limiting the rate at which policies change during each update step based on past experience, allowing for more stable convergence towards optimal solutions."



In [45]:
clean_answer = """Proximal Policy Optimization (PPO) is a reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability. It works by limiting excessive policy changes during updates using a clipped objective function, preventing significant deviations from the previous policy and avoiding the loss of valuable experience."""

results_clean = check_faithfulness(context_text, clean_answer)

for sentence, label, confidence in results_clean:
    flag = "POSSIBLE HALLUCINATION" if label != "entailment" else "supported"
    print(f"[{label.upper()} ({confidence:.2f})] {flag}")
    print(f"  \"{sentence}\"\n")

Checking the CLEAN (non-hallucinated) baseline answer:

[ENTAILMENT (0.99)] supported
  "Proximal Policy Optimization (PPO) is a reinforcement learning algorithm that enhances policy performance through iterative updates while ensuring training stability."

[ENTAILMENT (0.62)] supported
  "It works by limiting excessive policy changes during updates using a clipped objective function, preventing significant deviations from the previous policy and avoiding the loss of valuable experience."



Q5.3

Immplemented a check_faithfulness() function that splits a RAG-generated answer into
individual sentences and checks each one against the retrieved context using the NLI model,
treating the context as the *premise* and each answer sentence as the *hypothesis*.

Sentences
classified as entailment are considered grounded; neutral or contradiction sentences are
flagged as possible hallucinations.

**Validation on real data:** tested against an actual hallucinated PPO answer from Q4 (the
"Sam Altman" claim), the checker correctly flagged the fabricated sentence as NEUTRAL (97%
confidence) while correctly marking the mechanism-description sentence as ENTAILMENT (91%
confidence). Tested against a clean, non-hallucinated answer, both sentences were correctly
marked as ENTAILMENT (99% and 62% confidence), confirming the checker can tell the difference rather than flagging indiscriminately.